<a href="https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Debbie1236-cmd/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 4 — The Freshness Multiplier

The paper breaks old content into age groups and shows how much more "growing" content there is compared to "declining" content in each group:

31–90 days old: about 8 growing pages for every 1 declining page
181–360 days old: about 3 growing pages for every 1 declining page — this looks like a normal, believable pattern
361+ days old: jumps to roughly 28 growing pages for every 1 declining page — the biggest number on the page, and the one that makes refreshing old content look most powerful

My question: That last number, from the oldest bucket, is built on very little data — by my read, there's only one declining page in that whole group. If just one page had landed in a different category, that ratio could have looked completely different. A number that swings that easily because of one page isn't strong enough evidence to build a big claim on, even though it's the most dramatic number on the page.

Finding 2 — "What Predicts Growth?" (the AI model in the back of the paper)

Near the end of the paper, there's a section where they built an AI model to guess whether a page's traffic would grow or decline. They say it got the answer right 71% of the time when tested. To test it fairly, they had to split their data into two piles — one pile to teach the model, and a separate pile to test it on, so it's being tested on pages it's never seen before.

The methodology page says they used an "80/20 split" — meaning 80% of the data trained the model and 20% tested it. But it doesn't say how that split was made. The data comes from 57 different brands, and the paper never mentions whether all of one brand's pages were kept together on the same side of the split, or whether a single brand's pages could show up in both the training pile and the testing pile.

My question: If pages from the same brand ended up on both sides of the split, the model could partly be "recognizing" that brand's style rather than learning a general pattern that works on brand-new websites. That would make the 71% look better than it really is — the same way our own model scored higher (74%) before we split it properly by client, and dropped to a more honest 68% once no client repeated on both sides

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("https://raw.githubusercontent.com/Debbie1236-cmd/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = [
    "days_since_last_update", "avg_position", "has_position_data",
    "search_volume", "competition", "has_keyword_data",
    "ctr", "engagement_rate"
]

def build_features(d):
    d = d.copy()
    d["is_declining_label"] = (d["trend_direction"] == "down").astype(int)
    d["has_keyword_data"] = d["search_volume"].notna().astype(int)
    d["has_position_data"] = (d["avg_position"] != 0).astype(int)
    d["search_volume"] = d["search_volume"].fillna(0)
    d["competition"] = d["competition"].fillna(0)
    return d

def run_split(train_df, test_df, label):
    train_df, test_df = build_features(train_df), build_features(test_df)
    X_train, y_train = train_df[feature_cols], train_df["is_declining_label"]
    X_test, y_test = test_df[feature_cols], test_df["is_declining_label"]
    scaler = StandardScaler()
    Xtr, Xte = scaler.fit_transform(X_train), scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr, y_train)
    probs = model.predict_proba(Xte)[:, 1]
    p50, p100 = precision_at_k(probs, y_test.values, 50), precision_at_k(probs, y_test.values, 100)
    shared = set(train_df["client_id"]) & set(test_df["client_id"])
    print(f"[{label}] P@50={p50:.3f}  P@100={p100:.3f}  base_rate={y_test.mean():.3f}  shared_clients={len(shared)}")
    return p50, p100

# BEFORE: naive random split
train_rand, test_rand = train_test_split(df, test_size=0.2, random_state=42)
rand_p50, rand_p100 = run_split(train_rand, test_rand, "RANDOM split (before)")

# AFTER: client-grouped split
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
grp_p50, grp_p100 = run_split(df.iloc[train_idx], df.iloc[test_idx], "CLIENT-GROUPED split (after)")

print(f"\nGap at P@50: {rand_p50 - grp_p50:.3f}   Gap at P@100: {rand_p100 - grp_p100:.3f}")


[RANDOM split (before)] P@50=0.740  P@100=0.730  base_rate=0.545  shared_clients=31
[CLIENT-GROUPED split (after)] P@50=0.680  P@100=0.600  base_rate=0.511  shared_clients=0

Gap at P@50: 0.060   Gap at P@100: 0.130


Under a random split, precision@50 measured 0.74 — but 31 of 32 clients showed up in both training and testing, letting the model partly memorize client-specific patterns. Under a client-grouped split, where no client repeats on both sides, precision@50 measured 0.68 — the honest number.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [22]:
full_features = feature_cols
no_suspect = [f for f in full_features if f != "has_position_data"]

for name, cols in [("WITH has_position_data", full_features), ("WITHOUT has_position_data", no_suspect)]:
    train_df, test_df = build_features(df.iloc[train_idx]), build_features(df.iloc[test_idx])
    X_train, y_train = train_df[cols], train_df["is_declining_label"]
    X_test, y_test = test_df[cols], test_df["is_declining_label"]
    scaler = StandardScaler()
    Xtr, Xte = scaler.fit_transform(X_train), scaler.transform(X_test)
    m = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr, y_train)
    p = m.predict_proba(Xte)[:, 1]
    print(f"{name}: P@50={precision_at_k(p, y_test.values, 50):.3f}  P@100={precision_at_k(p, y_test.values, 100):.3f}")

# confirm trend_direction / trend_pct never entered the feature list
print("\nLabel-source columns kept out of features:", [c for c in feature_cols if "trend" in c] == [])


WITH has_position_data: P@50=0.680  P@100=0.600
WITHOUT has_position_data: P@50=0.640  P@100=0.610

Label-source columns kept out of features: True


Removing has_position_data dropped precision@50 from 0.68 to 0.64 — a small, real decrease, not a collapse toward random guessing. This means the feature reflects a genuine pattern (tracked pages behave very differently from untracked ones), not leaked label information. The columns the label was built from (trend_direction, trend_pct) were never included as features.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [23]:
before = "My logistic regression clearly beats the baseline... a meaningful, honest improvement, not a marginal one."
after = ("The observed precision@50 was 0.68, measured under a client-grouped split — a directional "
         "improvement over the 0.40 baseline, useful as decision-support for prioritizing review, "
         "not a guarantee on a brand-new client.")
print("BEFORE:", before)
print("AFTER:", after)

BEFORE: My logistic regression clearly beats the baseline... a meaningful, honest improvement, not a marginal one.
AFTER: The observed precision@50 was 0.68, measured under a client-grouped split — a directional improvement over the 0.40 baseline, useful as decision-support for prioritizing review, not a guarantee on a brand-new client.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.